# Week 32, Day 5: Protocols

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w032d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import io
from typing import Protocol, runtime_checkable

@runtime_checkable
class Writer(Protocol):
    def write(self, text: str) -> int: ...

class Screen:
    def write(self, text):
        print(text, end="")
        return len(text)

class Kettle:
    def boil(self):
        return "whee"

print(isinstance(Screen(), Writer), isinstance(io.StringIO(), Writer), isinstance(Kettle(), Writer))

In [ ]:
from typing import Protocol

class Writer(Protocol):
    """Anything that text can be written to."""

    def write(self, text: str) -> int:
        """Write text and return how many characters were written."""
        ...

def print_receipt(items, out: Writer):
    total = 0
    for name, pence in items:
        out.write(f"{name:<10}{pence / 100:>7.2f}\n")
        total += pence
    out.write(f"{'Total':<10}{total / 100:>7.2f}\n")

class Screen:
    def write(self, text):
        print(text, end="")
        return len(text)

print_receipt([("Suya", 450), ("Zobo", 180)], Screen())

In [ ]:
from typing import Protocol, runtime_checkable

class Plain(Protocol):
    def close(self): ...

@runtime_checkable
class Closable(Protocol):
    def close(self): ...

class Door:
    def close(self):
        return "shut"

try:
    isinstance(Door(), Plain)
except TypeError as err:
    print("TypeError:", err)
print(isinstance(Door(), Closable), isinstance("door", Closable))

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Writer(Protocol):
    def write(self, text: str) -> int: ...

class Log:
    def __init__(self, out):
        if not isinstance(out, Writer):
            raise TypeError(f"Log needs something with write(), not {type(out).__name__}")
        self.out = out

    def info(self, message):
        self.out.write(f"[INFO] {message}\n")

try:
    Log("log.txt")
except TypeError as err:
    print("TypeError:", err)
with open("log.txt", "w") as f:
    Log(f).info("started")
print(open("log.txt").read(), end="")

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class HasArea(Protocol):
    def area(self) -> float: ...

class Square:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

class Field:
    area = 4000                       # a number, not a method

class Rug:
    def area(self, units):            # needs an argument the Protocol doesn't give
        return 6

for thing in [Square(3), Field(), Rug()]:
    print(type(thing).__name__, isinstance(thing, HasArea))

In [ ]:
from typing import Protocol, runtime_checkable
from types import SimpleNamespace

@runtime_checkable
class Plugin(Protocol):
    name: str
    def run(self, text: str) -> str: ...

shout = SimpleNamespace(name="shout", run=lambda text: text.upper())
print(isinstance(shout, Plugin), shout.run("ẹ káàbọ̀"))

In [ ]:
from fractions import Fraction
from decimal import Decimal
from typing import SupportsFloat

def mean(values):
    for v in values:
        if not isinstance(v, SupportsFloat):
            raise TypeError(f"can't average a {type(v).__name__}")
    return sum(float(v) for v in values) / len(values)

print(mean([1, 2.5, Fraction(1, 2), Decimal("4")]))
try:
    mean([1, "2"])
except TypeError as err:
    print("TypeError:", err)

> **Key idea.** Nominal typing asks what a class inherits from; structural typing asks what methods it has. A Protocol (`class Writer(Protocol):` with methods whose body is `...`) describes an interface structurally: anything with matching methods fits, without inheriting. Annotations like `text: str` and `-> int` document types but aren't checked when the program runs; type checkers (Week 33) check them. `@runtime_checkable` lets `isinstance` test a Protocol, but only for the presence of the names, not their kind or signatures. `collections.abc` and `typing.Supports...` provide ready-made Protocols. Use a Protocol for "anything that can do X", an ABC for a family you design.

### Exercise: Closing anything

Write a runtime-checkable Protocol `Closable` with a method `close()`, and a function `close_all(things)` that calls `close()` on every item that matches `Closable`, skips the rest, and returns how many it closed.

In [ ]:
from typing import Protocol, runtime_checkable

___

def close_all(things):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_close():\n    \"closing\"\n    import io\n    C = ns[\"Closable\"]\n    class Door:\n        def __init__(self):\n            self.shut = False\n        def close(self):\n            self.shut = True\n    d, s = Door(), io.StringIO()\n    assert close_all([d, \"text\", 3, s, None]) == 2 and d.shut and s.closed\n    assert close_all([]) == 0 and isinstance(d, C) and not isinstance(5, C)")

### Exercise: Averaging numbers of any kind

Write `average(values)`, the mean of the values as a `float`, rounded to 3 places. It accepts anything that matches `typing.SupportsFloat` (ints, floats, `Fraction`, `Decimal`, your own classes with `__float__`), and raises `TypeError` naming the type of the first value that doesn't. An empty list raises `ValueError`.

In [ ]:
from typing import SupportsFloat

def average(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_average():\n    \"averages\"\n    from fractions import Fraction\n    from decimal import Decimal\n    class Grams:\n        def __init__(self, g):\n            self.g = g\n        def __float__(self):\n            return float(self.g)\n    assert average([1, 2]) == 1.5 and average([Fraction(1, 3), Fraction(1, 3)]) == 0.333 and average([Decimal(\"2.5\"), Grams(4)]) == 3.25\n    for bad in [[1, \"2\"], [None], [[1]]]:\n        try:\n            average(bad)\n            assert False, f\"{bad} should raise TypeError\"\n        except TypeError as err:\n            assert type(bad[-1]).__name__ in str(err), f\"the message should name the type: {err}\"\n    try:\n        average([])\n        assert False, \"an empty list should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: A logger that writes anywhere

Write a runtime-checkable Protocol `Sink` with `write(text)`. Then write:

- `Logger(sink, level="INFO")`: raises `TypeError` if `sink` doesn't match `Sink`. `log(level, message)` writes `"[WARNING] disk nearly full\n"` to the sink if `level` is at or above the logger's level (the levels in order: `DEBUG`, `INFO`, `WARNING`, `ERROR`; an unknown level raises `ValueError`), and returns True if it wrote.
- `Tee(*sinks)`: a sink that writes everything to each of several sinks (each must match `Sink`, otherwise `TypeError`). A `Tee` is itself a `Sink`, so it can be given to a `Logger`, or to another `Tee`.

In [ ]:
from typing import Protocol, runtime_checkable

LEVELS = ["DEBUG", "INFO", "WARNING", "ERROR"]

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_logger():\n    \"logging\"\n    import io\n    L = ns[\"Logger\"]\n    buf = io.StringIO()\n    log = L(buf, level=\"WARNING\")\n    assert log.log(\"INFO\", \"quiet\") is False and log.log(\"ERROR\", \"disk failed\") is True and log.log(\"WARNING\", \"slow\") is True\n    assert buf.getvalue() == \"[ERROR] disk failed\\n[WARNING] slow\\n\"\n    for bad in [lambda: L(\"file.txt\"), lambda: L(None)]:\n        try:\n            bad()\n            assert False, \"a non-sink should raise TypeError\"\n        except TypeError:\n            pass\n    for bad in [lambda: L(buf, level=\"LOUD\"), lambda: log.log(\"FATAL\", \"x\")]:\n        try:\n            bad()\n            assert False, \"an unknown level should raise ValueError\"\n        except ValueError:\n            pass\ndef test_tee():\n    \"tees\"\n    import io\n    L, T, S = ns[\"Logger\"], ns[\"Tee\"], ns[\"Sink\"]\n    a, b, c = io.StringIO(), io.StringIO(), io.StringIO()\n    t = T(a, T(b, c))\n    assert isinstance(t, S)\n    L(t).log(\"INFO\", \"hello\")\n    assert a.getvalue() == b.getvalue() == c.getvalue() == \"[INFO] hello\\n\"\n    try:\n        T(a, \"b\")\n        assert False, \"a Tee of non-sinks should raise TypeError\"\n    except TypeError:\n        pass")

### Exercise: Plugins without a parent class

Write a runtime-checkable Protocol `Plugin` with an attribute `name: str` and a method `run(text: str) -> str`, and a `Registry`:

- `register(plugin)` adds a plugin (raising `TypeError` if it doesn't match `Plugin`, and `ValueError` if a plugin with that name is already registered) and returns it;
- `names()` lists the names in the order registered;
- `run_all(text)` passes the text through every plugin in order and returns the result.

Plugins can be objects of any class, or even `types.SimpleNamespace` objects with a `name` and a `run` function.

In [ ]:
from typing import Protocol, runtime_checkable

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_registry():\n    \"plugins\"\n    from types import SimpleNamespace\n    R, P = ns[\"Registry\"], ns[\"Plugin\"]\n    class Strip:\n        name = \"strip\"\n        def run(self, text):\n            return text.strip()\n    reg = R()\n    reg.register(Strip())\n    reg.register(SimpleNamespace(name=\"shout\", run=lambda t: t.upper() + \"!\"))\n    assert reg.names() == [\"strip\", \"shout\"] and reg.run_all(\"  oya  \") == \"OYA!\" and R().run_all(\"same\") == \"same\"\n    for bad in [\"plugin\", SimpleNamespace(name=\"x\"), SimpleNamespace(run=str.upper)]:\n        try:\n            reg.register(bad)\n            assert False, f\"{bad!r} isn't a plugin\"\n        except TypeError:\n            pass\n    try:\n        reg.register(SimpleNamespace(name=\"strip\", run=str.lower))\n        assert False, \"a duplicate name should raise ValueError\"\n    except ValueError:\n        pass\n    assert reg.names() == [\"strip\", \"shout\"]")

### Bug hunt: Can't check that

The check crashes with `TypeError: Instance and class checks can only be used with @runtime_checkable protocols`. Fix the Protocol so it prints `True False`.

In [ ]:
from typing import Protocol, runtime_checkable

class Drawable(Protocol):
    def draw(self) -> str: ...

class Star:
    def draw(self):
        return "*"

print(isinstance(Star(), Drawable), isinstance("*", Drawable))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"checked\"\n    expect_output(\"True False\")")

### Bug hunt: Not a sink

The text-message sender is refused with `TypeError: not a sink: SmsSender`, because its method has the wrong name for the Protocol. Fix `SmsSender` (not the Protocol) so this prints `SMS: [INFO] hello`.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Sink(Protocol):
    def write(self, text: str): ...

class SmsSender:
    def send(self, text):
        print("SMS:", text)

def log(sink, message):
    if not isinstance(sink, Sink):
        raise TypeError(f"not a sink: {type(sink).__name__}")
    sink.write(f"[INFO] {message}")

log(SmsSender(), "hello")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sent\"\n    expect_output(\"SMS: [INFO] hello\")")

### Bug hunt: It passed the check

`Plot` passes the `isinstance` check, but the total crashes with `TypeError: 'int' object is not callable`. The Protocol is right; fix `Plot` so it fits it properly, and the program prints `69`.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class HasArea(Protocol):
    def area(self) -> float: ...

class Room:
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h

class Plot:
    def __init__(self, w, h):
        self.w, self.h = w, h

    @property
    def area(self):
        return self.w * self.h

things = [Room(3, 4), Plot(5, 9), Room(2, 6)]
assert all(isinstance(t, HasArea) for t in things)
print(sum(t.area() for t in things))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the total\"\n    expect_output(\"69\")")

### Puzzle: A drawable Protocol

Arrange the lines into a runtime-checkable Protocol and a class that matches it without inheriting. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from typing import Protocol, runtime_checkable, SupportsInt
from collections.abc import Sized

@runtime_checkable
class Speaker(Protocol):
    def speak(self) -> str: ...

class Dog:
    def speak(self):
        return "woof"

class Statue:
    speak = "..."

class Crowd:
    def __len__(self):
        return 3000

p = ___   # isinstance(Dog(), Speaker)
q = ___   # isinstance(Statue(), Speaker)
r = ___   # isinstance(Crowd(), Sized)
s = ___   # isinstance(2.9, SupportsInt)
t = ___   # isinstance("9", SupportsInt)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (True, True, True), \"p to r\"\n    assert (s, t) == (True, False)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: one checkout, many ways to pay

A shop's checkout should accept payment methods from different suppliers, whose classes it can't change. Two are given in the starter, with interfaces that don't match each other. Make them fit one Protocol with small **adapter** classes (each holds the supplier's object and translates calls).

- `PaymentFailed(Exception)`.
- `PaymentMethod`, a runtime-checkable Protocol with an attribute `name: str` and a method `charge(pence: int) -> str` that returns a reference, or raises `PaymentFailed`.
- `CardAdapter(card)`: `name` is `"card"`. `charge(pence)` calls `card.pay_amount(pounds)` (the amount in pounds, as a float), which returns a dictionary like `{"ok": True, "ref": "C-1"}`; if `ok` is False, raise `PaymentFailed` with the dictionary's `"reason"`.
- `WalletAdapter(wallet)`: `name` is `"wallet"`. `charge(pence)` calls `wallet.transfer(pence)`, which returns a reference or raises `RuntimeError`; turn that into `PaymentFailed` with the same message.
- `Checkout(methods)`: every method must match `PaymentMethod` (otherwise `TypeError`). `pay(pence, prefer=None)` checks that `pence` is an `int` of 1 or more (`ValueError`), then tries the methods in order, or the one named `prefer` first (an unknown name raises `ValueError`), and returns `(name, reference)` from the first that succeeds. If all fail, raise `PaymentFailed` with a message listing each method's name and reason, like `card: declined; wallet: no funds`.

In [ ]:
from typing import Protocol, runtime_checkable

class LegacyCard:
    """A supplier's card terminal. Don't change it."""
    def __init__(self, limit_pounds):
        self.limit = limit_pounds
        self.count = 0
    def pay_amount(self, pounds):
        if pounds > self.limit:
            return {"ok": False, "reason": "declined"}
        self.count += 1
        return {"ok": True, "ref": f"C-{self.count}"}

class MobileWallet:
    """Another supplier's wallet. Don't change it either."""
    def __init__(self, balance_pence):
        self.balance = balance_pence
    def transfer(self, pence):
        if pence > self.balance:
            raise RuntimeError("no funds")
        self.balance -= pence
        return f"W{self.balance}"

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def setup():\n    return ns[\"CardAdapter\"](ns[\"LegacyCard\"](20)), ns[\"WalletAdapter\"](ns[\"MobileWallet\"](5000))\ndef test_adapters():\n    \"the adapters\"\n    PM, PF = ns[\"PaymentMethod\"], ns[\"PaymentFailed\"]\n    card, wallet = setup()\n    assert isinstance(card, PM) and isinstance(wallet, PM) and (card.name, wallet.name) == (\"card\", \"wallet\")\n    assert card.charge(1999) == \"C-1\" and wallet.charge(1500) == \"W3500\"\n    for method, amount, reason in [(card, 2001, \"declined\"), (wallet, 3501, \"no funds\")]:\n        try:\n            method.charge(amount)\n            assert False, \"should raise PaymentFailed\"\n        except PF as err:\n            assert str(err) == reason, f\"message was {str(err)!r}\"\ndef test_checkout():\n    \"the checkout\"\n    from types import SimpleNamespace\n    C, PF = ns[\"Checkout\"], ns[\"PaymentFailed\"]\n    card, wallet = setup()\n    co = C([card, wallet])\n    assert co.pay(1000) == (\"card\", \"C-1\") and co.pay(4000) == (\"wallet\", \"W1000\") and co.pay(500, prefer=\"wallet\") == (\"wallet\", \"W500\")\n    try:\n        co.pay(9000)\n        assert False, \"everything should fail\"\n    except PF as err:\n        assert str(err) == \"card: declined; wallet: no funds\", f\"message was {str(err)!r}\"\n    free = SimpleNamespace(name=\"voucher\", charge=lambda pence: \"V-1\")\n    assert C([free]).pay(10**6) == (\"voucher\", \"V-1\")\n    for bad in [0, -5, 2.5, True]:\n        try:\n            co.pay(bad)\n            assert False, f\"pay({bad!r}) should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        co.pay(100, prefer=\"cash\")\n        assert False, \"an unknown method should raise ValueError\"\n    except ValueError:\n        pass\n    for bad in [[ns[\"LegacyCard\"](5)], [\"card\"]]:\n        try:\n            C(bad)\n            assert False, \"a non-PaymentMethod should raise TypeError\"\n        except TypeError:\n            pass")

## Warm-up

Neither `Screen` nor `io.StringIO` inherits from `Writer`, and `Writer` was written after them, yet both count as writers: each has a `write` method. A `Kettle` doesn't. That's a Protocol: an interface defined by **what an object can do**, not by what it inherits from.

## Two ways to belong

There are two ways for an object to "be a" something:

| | Nominal (by name) | Structural (by shape) |
|---|---|---|
| An object belongs if... | its class **inherits** from the type | its class **has the methods** the type describes |
| Python tool | classes, ABCs (Day 4) | duck typing (Week 30), Protocols (today) |
| Example | a `Card` is a `Payment` because it says `class Card(Payment)` | a `Screen` is a `Writer` because it has `write` |
| Good for | families you design together | code that should accept things written by anyone, including the standard library |

Duck typing has always been structural, but until now the interface only lived in docstrings ("anything with a `write(text)` method"). A Protocol writes it down as a class, so people (and, from Week 33, checking tools) can read it.

## Writing a Protocol

A Protocol is a class that inherits from `typing.Protocol` and lists methods with **no real bodies**, just `...`:

Two new pieces of notation:

- `text: str` on a **parameter**, and `-> int` after the brackets for the **return value**, are annotations, like the field annotations of Day 1. They say what types are expected. Python itself doesn't check them.
- `out: Writer` says `print_receipt` needs something that matches the `Writer` Protocol.

Notice that `Screen` doesn't mention `Writer` anywhere. It doesn't have to. That's the whole point: a Protocol describes what's needed, and anything that fits, fits, including classes you didn't write and can't change, like `io.StringIO` or an open file.

So what does writing `out: Writer` actually **do** when the program runs? Nothing at all; pass a `Kettle` and it still fails with `AttributeError` inside the function. Protocols do their real work in **type checkers**, tools that read your code before it runs and report calls that don't fit. Week 33 uses one, `mypy`. Until then, a Protocol is precise documentation, and with one more step it can be checked at run time.

## Checking at run time, and its limits

Put `@runtime_checkable` above a Protocol and `isinstance` works with it. Without the decorator, `isinstance` refuses:

This is useful at the edges of a program, to give a clear error early instead of a confusing one later:

But know exactly what the check tests: **only that attributes with those names exist**. It doesn't check that they're methods, how many arguments they take, or what they return:

All three pass, though only `Square` really fits. A runtime check is a quick sanity check, not a proof; the thorough checking is the type checker's job (Week 33). Two more limits: a Protocol can't be made into an object itself (`Writer()` raises `TypeError`), and `issubclass` doesn't work with Protocols that list data attributes.

Protocols can list **attributes** as well as methods, written like dataclass fields:

`types.SimpleNamespace` makes a quick object with whatever attributes you give it, so this "plugin" has no class of its own at all, and still matches.

## Protocols you already have

The standard library is full of Protocols, though some of them predate the name:

| Protocol | Where | An object matches if it has |
|---|---|---|
| `Iterable`, `Sized`, `Container`, `Hashable`, `Callable` | `collections.abc` | `__iter__`, `__len__`, `__contains__`, `__hash__`, `__call__` |
| `SupportsInt`, `SupportsFloat`, `SupportsIndex` | `typing` | `__int__`, `__float__`, `__index__` |
| `SupportsAbs`, `SupportsRound` | `typing` | `__abs__`, `__round__` |

Day 4 showed that the `collections.abc` ones work with `isinstance` without inheriting: that's structural checking, exactly what a runtime-checkable Protocol does. The `typing` ones are runtime-checkable too:

Week 31's special methods are what these Protocols look for. A class with `__float__` is already a `SupportsFloat`, without knowing the name exists.

## Protocol or ABC?

| Question | If yes |
|---|---|
| Should classes you don't control (the standard library's, other people's) count? | Protocol |
| Is it a family you design together, where children should inherit shared code? | ABC |
| Do you want Python to refuse to make an incomplete object? | ABC |
| Is the interface one or two methods that many unrelated things might have? | Protocol |
| Are you mostly documenting what a function needs from its argument? | Protocol |

You can also have both: a class may inherit from a Protocol explicitly (`class Screen(Writer):`), which documents the intent and lets it use any ordinary methods the Protocol defines. But it never has to.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a function in your own code whose docstring says something like "anything with a `read()` method", and write that requirement as a Protocol. Tomorrow is the Forge: dataclasses, Enums, named tuples, ABCs and Protocols, working together.